In [ ]:
import torch
import sys
import os
from formerlens import HookedOneLayerTransformer
import circuitsvis as cv
import numpy as np


/Users/gabesmithline/.pyenv/versions/3.13.2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = HookedOneLayerTransformer.from_checkpoint("checkpoints/best_model.pt")
model.eval()

print(f"Model: {model.model_dim}d, {model.num_heads} heads, vocab={model.vocab_size}")


Model: 64d, 4 heads, vocab=12


## Token Reference
- Digits 0-9 → tokens 0-9
- EOS = 11
- Input format: reversed digits + EOS (e.g., 42 → [2, 4, 11])


In [3]:
EOS_TOKEN = 11

def num_to_input(n, num_digits=2):
    """Convert number to model input tokens."""
    digits = []
    for _ in range(num_digits):
        digits.append(n % 10)
        n //= 10
    return digits + [EOS_TOKEN]

def tokens_to_labels(tokens):
    """Convert tokens to string labels."""
    return [str(t) if t < 10 else "EOS" if t == 11 else "PAD" for t in tokens]


## Visualize Attention Pattern


In [ ]:
# Test: 42 + 7 = 49
test_num = 42
tokens = torch.tensor([num_to_input(test_num)])
token_labels = tokens_to_labels(tokens[0].tolist())

print(f"Input: {test_num} → {token_labels}")

# Get attention patterns
logits, cache = model.run_with_cache(tokens)
attn_pattern = cache["atn.hook_pattern"]  # (batch, heads, query, key)

# Prediction
pred = logits[0, -1].argmax().item()
expected = (test_num + 7) % 10 
print(f"Predicts: {pred} (expected: {expected}) {'✓' if pred == expected else '✗'}")


Input: 42 → ['2', '4', 'EOS']
Predicts: 9 (expected: 9) ✓


In [ ]:
cv.attention.attention_patterns(
    tokens=token_labels,
    attention=attn_pattern[0],  # Remove batch dim: (heads, query, key)
)


## Attention Heads View


In [6]:
# Attention heads visualization (shows what each head attends to)
cv.attention.attention_heads(
    tokens=token_labels,
    attention=attn_pattern[0],
)


## Compare Multiple Inputs


In [7]:
# Test multiple numbers
test_cases = [12, 42, 73, 99]

for num in test_cases:
    tokens = torch.tensor([num_to_input(num)])
    labels = tokens_to_labels(tokens[0].tolist())
    
    logits, cache = model.run_with_cache(tokens)
    attn = cache["atn.hook_pattern"]
    
    pred = logits[0, -1].argmax().item()
    expected = (num + 7) % 10
    status = "✓" if pred == expected else "✗"
    
    print(f"\n{num} + 7 = {num + 7} | Pred: {pred} {status}")
    display(cv.attention.attention_patterns(tokens=labels, attention=attn[0]))



12 + 7 = 19 | Pred: 9 ✓



42 + 7 = 49 | Pred: 9 ✓



73 + 7 = 80 | Pred: 0 ✓



99 + 7 = 106 | Pred: 6 ✓


## Attention During Generation


In [13]:
def generate_with_viz(model, input_num, max_steps=4):
    """Generate and visualize attention at each step."""
    tokens = torch.tensor([num_to_input(input_num)])
    generated = []
    
    for step in range(max_steps):
        labels = tokens_to_labels(tokens[0].tolist())
        logits, cache = model.run_with_cache(tokens)
        attn = cache["atn.hook_pattern"]
        
        next_token = logits[0, -1].argmax().item()
        generated.append(next_token)
        
        print(f"\nStep {step + 1}: seq={labels}, next={next_token}")
        display(cv.attention.attention_patterns(tokens=labels, attention=attn[0]))
        
        if next_token == EOS_TOKEN:
            break
        tokens = torch.cat([tokens, torch.tensor([[next_token]])], dim=1)
    
    return generated

# Generate 42 + 7 = 49
print("Generating output for 42 + 7:")
result = generate_with_viz(model, 42)
print(f"\nGenerated: {result} (expected: [9, 4, 0, 11])")


Generating output for 42 + 7:

Step 1: seq=['2', '4', 'EOS'], next=9



Step 2: seq=['2', '4', 'EOS', '9'], next=4



Step 3: seq=['2', '4', 'EOS', '9', '4'], next=0



Step 4: seq=['2', '4', 'EOS', '9', '4', '0'], next=11



Generated: [9, 4, 0, 11] (expected: [9, 4, 0, 11])


## Attention Head Summary


In [9]:
# Visualize attention for head analysis
tokens = torch.tensor([num_to_input(42)])
labels = tokens_to_labels(tokens[0].tolist())
_, cache = model.run_with_cache(tokens)
attn = cache["atn.hook_pattern"]

# Attention heads with names
cv.attention.attention_heads(
    tokens=labels,
    attention=attn[0],
    attention_head_names=[f"Head {i}" for i in range(model.num_heads)],
)


## Head Specialization Analysis


In [ ]:
num_samples = 100
head_focus = {h: {"ones": [], "tens": [], "eos": []} for h in range(model.num_heads)}

for _ in range(num_samples):
    num = torch.randint(0, 100, (1,)).item()
    tokens = torch.tensor([num_to_input(num)])
    _, cache = model.run_with_cache(tokens)
    attn = cache["atn.hook_pattern"][0, :, -1, :].detach().numpy()  # Last query position
    
    for h in range(model.num_heads):
        head_focus[h]["ones"].append(attn[h, 0])
        head_focus[h]["tens"].append(attn[h, 1])
        head_focus[h]["eos"].append(attn[h, 2])

print("Head Specialization (attention at last position, averaged over 100 samples):")
print("="*60)
for h in range(model.num_heads):
    ones = np.mean(head_focus[h]["ones"])
    tens = np.mean(head_focus[h]["tens"])
    eos = np.mean(head_focus[h]["eos"])
    
    focus = max([("ones digit", ones), ("tens digit", tens), ("EOS", eos)], key=lambda x: x[1])
    
    print(f"Head {h}: ones={ones:.2f}, tens={tens:.2f}, EOS={eos:.2f} → focuses on {focus[0]}")


Head Specialization (attention at last position, averaged over 100 samples):
Head 0: ones=0.43, tens=0.00, EOS=0.57 → focuses on EOS
Head 1: ones=0.45, tens=0.01, EOS=0.54 → focuses on EOS
Head 2: ones=0.00, tens=0.00, EOS=1.00 → focuses on EOS
Head 3: ones=0.71, tens=0.00, EOS=0.28 → focuses on ones digit
